# 16 — Liquidity & Stop Hunts

## Goal
Identify **liquidity pools** (clusters of retail stops) and detect **stop-hunt wicks** — the spike into that pool followed by a reversal.  
A stop hunt at the proximal line of a zone is one of the highest-probability entry triggers.

---

## Swing-point Liquidity

Retail traders place stops just below obvious swing lows (for longs) and above obvious swing highs (for shorts).  
These clusters create **liquidity** that institutions must consume before a strong move.

A **stop hunt** occurs when:
1. Price spikes to or beyond a swing point.
2. The candle closes back inside the range — the wick is long relative to the body.

$$\text{hunt\_wick} = \frac{\text{wick length beyond swing}}{\text{ATR}} \geq 0.5$$

---

## Detection

For each swing low `sl` in the dataset:

$$\text{hunt} = \exists\, i : \text{low}_i < \text{sl} \;\text{ and }\; \text{close}_i > \text{sl} \;\text{ and }\; \frac{\text{sl} - \text{low}_i}{\text{ATR}_i} \geq 0.5$$

Mirror logic for swing highs (supply hunts).

---

## Visualization
Swing points marked with triangles; stop-hunt candles highlighted with a coloured border.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. Detect swing points and stop-hunt wicks

In [ ]:
SWING_WINDOW = 3
HUNT_ATR_MIN = 0.5

def find_swings():
    sh_idx, sl_idx = [], []
    for i in range(SWING_WINDOW, len(h_arr) - SWING_WINDOW):
        if h_arr[i] == h_arr[i-SWING_WINDOW:i+SWING_WINDOW+1].max(): sh_idx.append(i)
        if l_arr[i] == l_arr[i-SWING_WINDOW:i+SWING_WINDOW+1].min(): sl_idx.append(i)
    return sh_idx, sl_idx

sh_idx, sl_idx = find_swings()

demand_hunts, supply_hunts = [], []

for sl_i in sl_idx:
    sl_price = l_arr[sl_i]
    for i in range(sl_i + 1, min(sl_i + 10, len(l_arr))):
        if l_arr[i] < sl_price and c_arr[i] > sl_price:
            wick = (sl_price - l_arr[i]) / atr_arr[i] if atr_arr[i] > 0 else 0
            if wick >= HUNT_ATR_MIN:
                demand_hunts.append({"bar": i, "swing": sl_i, "level": sl_price, "wick_atr": round(wick,2)})
                break

for sh_i in sh_idx:
    sh_price = h_arr[sh_i]
    for i in range(sh_i + 1, min(sh_i + 10, len(h_arr))):
        if h_arr[i] > sh_price and c_arr[i] < sh_price:
            wick = (h_arr[i] - sh_price) / atr_arr[i] if atr_arr[i] > 0 else 0
            if wick >= HUNT_ATR_MIN:
                supply_hunts.append({"bar": i, "swing": sh_i, "level": sh_price, "wick_atr": round(wick,2)})
                break

print(f"Demand stop-hunts: {len(demand_hunts)}")
for h in demand_hunts:
    print(f"  bar={h['bar']}  swing={h['swing']}  level={h['level']:.2f}  wick_atr={h['wick_atr']}")
print(f"Supply stop-hunts: {len(supply_hunts)}")
for h in supply_hunts:
    print(f"  bar={h['bar']}  swing={h['swing']}  level={h['level']:.2f}  wick_atr={h['wick_atr']}")


## 3. Visualization — swing points and stop-hunt wicks

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
))

# swing highs / lows
fig.add_trace(go.Scatter(x=df.index[sh_idx], y=h_arr[sh_idx],
    mode="markers", marker=dict(symbol="triangle-up", size=7, color="#26a69a"),
    name="Swing High"))
fig.add_trace(go.Scatter(x=df.index[sl_idx], y=l_arr[sl_idx],
    mode="markers", marker=dict(symbol="triangle-down", size=7, color="#ef5350"),
    name="Swing Low"))

# stop-hunt markers
for h in demand_hunts:
    fig.add_annotation(x=df.index[h["bar"]], y=l_arr[h["bar"]],
                       text=f"▲hunt {h['wick_atr']}", showarrow=True,
                       arrowhead=2, arrowcolor="#26a69a",
                       font=dict(size=8, color="#26a69a"), yanchor="top")

for h in supply_hunts:
    fig.add_annotation(x=df.index[h["bar"]], y=h_arr[h["bar"]],
                       text=f"▼hunt {h['wick_atr']}", showarrow=True,
                       arrowhead=2, arrowcolor="#ef5350",
                       font=dict(size=8, color="#ef5350"), yanchor="bottom")

fig.update_layout(
    title="Liquidity & Stop Hunts",
    height=540, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
